In [13]:
# Read large .bin file safely in chunks as uint8 numbers
with open("/home/vis/Desk/MyVault/Semester 5/GPU Programming/project/BANG-Billion-Scale-ANN/BANG_Exactdistance/build/sift10k_groundtruth.bin", "rb") as f:
    chunk_size = 4096  # 4 KB at a time
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break

        # Convert bytes to list of unsigned 8-bit integers
        numbers = list(chunk)
        print(numbers)


[16, 39, 0, 0, 10, 0, 0, 0, 128, 8, 0, 0, 168, 14, 0, 0, 114, 3, 0, 0, 169, 15, 0, 0, 21, 11, 0, 0, 190, 0, 0, 0, 31, 14, 0, 0, 48, 3, 0, 0, 21, 4, 0, 0, 92, 7, 0, 0, 221, 10, 0, 0, 102, 37, 0, 0, 188, 9, 0, 0, 42, 5, 0, 0, 64, 12, 0, 0, 14, 4, 0, 0, 92, 37, 0, 0, 157, 3, 0, 0, 158, 15, 0, 0, 135, 8, 0, 0, 147, 10, 0, 0, 210, 38, 0, 0, 138, 10, 0, 0, 244, 38, 0, 0, 83, 27, 0, 0, 145, 26, 0, 0, 202, 34, 0, 0, 112, 20, 0, 0, 18, 24, 0, 0, 79, 20, 0, 0, 115, 38, 0, 0, 97, 38, 0, 0, 102, 37, 0, 0, 110, 37, 0, 0, 1, 16, 0, 0, 104, 37, 0, 0, 109, 37, 0, 0, 16, 1, 0, 0, 103, 37, 0, 0, 0, 16, 0, 0, 111, 18, 0, 0, 44, 20, 0, 0, 135, 6, 0, 0, 2, 6, 0, 0, 9, 23, 0, 0, 156, 18, 0, 0, 207, 17, 0, 0, 102, 1, 0, 0, 143, 22, 0, 0, 14, 18, 0, 0, 73, 4, 0, 0, 215, 4, 0, 0, 79, 19, 0, 0, 155, 12, 0, 0, 36, 3, 0, 0, 47, 10, 0, 0, 220, 15, 0, 0, 91, 17, 0, 0, 150, 16, 0, 0, 40, 12, 0, 0, 152, 9, 0, 0, 197, 11, 0, 0, 146, 6, 0, 0, 133, 33, 0, 0, 214, 10, 0, 0, 202, 13, 0, 0, 156, 3, 0, 0, 172, 10, 0, 0, 229

In [10]:
import numpy as np
import struct # Import struct to read integers alongside floats

# Open the file in binary read mode
with open('build/siftsmall_query.bin', 'rb') as f:
    # Read the first 4 bytes (number of vectors) and ignore it
    num_vectors_header = struct.unpack('<i', f.read(4))[0] 
    # '<i' means little-endian signed integer
    
    # Read the next 4 bytes (dimension)
    d = struct.unpack('<i', f.read(4))[0]
    
    # Calculate the expected number of floats for the remaining data
    # (Total bytes - 8 header bytes) / 4 bytes per float
    remaining_bytes = f.seek(0, 2) - 8 # f.seek(0, 2) gets file size
    expected_floats = remaining_bytes // 4
    
    # Go back to the beginning of the vector data (offset 8)
    f.seek(8) 
    
    # Read the rest of the file as floats
    vector_data = np.fromfile(f, dtype=np.float32, count=expected_floats)
    
    # Reshape based on the dimension d
    # The number of rows will be calculated automatically (-1)
    try:
        sift_queries = vector_data.reshape(-1, d)
        
        print(f"File Size (bytes): {remaining_bytes + 8}")
        print(f"Dimension read: {d}")
        # print(f"Num vectors in header: {num_vectors_header}") # This is likely wrong (10000)
        print(f"Actual num vectors loaded: {sift_queries.shape[0]}") # Should be 100
        print(f"Loaded {sift_queries.shape[0]} queries of dimension {sift_queries.shape[1]}")
    
    except ValueError as e:
        print(f"Error reshaping data!")
        print(f"Total floats read after header: {len(vector_data)}")
        print(f"Dimension read: {d}")
        print(f"Is {len(vector_data)} divisible by {d}? {len(vector_data) % d == 0}")
        print(e)

# Example output expected:
# File Size (bytes): 51208
# Dimension read: 128
# Actual num vectors loaded: 100
# Loaded 100 queries of dimension 128

File Size (bytes): 51208
Dimension read: 128
Actual num vectors loaded: 100
Loaded 100 queries of dimension 128


In [ ]:
import numpy as np
import struct

try:
    with open('build/sift10k_groundtruth.bin', 'rb') as f:
        # Read the number of queries (N)
        num_queries = struct.unpack('<i', f.read(4))[0]
        # Read the number of neighbors per query (k)
        neighbors_per_query = struct.unpack('<i', f.read(4))[0]

        print(f"--- sift10k_groundtruth.bin ---")
        print(f"Header: N={num_queries}, k={neighbors_per_query}")

        # Define the structured data type for one record: (ID, Distance)
        # '<i4' = little-endian 4-byte signed integer (ID)
        # '<f4' = little-endian 4-byte float (Distance)
        record_dtype = np.dtype([('id', '<i4'), ('distance', '<f4')])
        
        # Calculate the number of records (pairs) to read
        expected_records = num_queries * neighbors_per_query
        
        # Read the rest of the file using this structured dtype
        data = np.fromfile(f, dtype=record_dtype, count=expected_records)
        
        # Reshape the 1D array of pairs into (N, k)
        ground_truth_pairs = data.reshape(num_queries, neighbors_per_query)

        # Now you can access the IDs and distances separately
        neighbor_ids = ground_truth_pairs['id']
        distances = ground_truth_pairs['distance']

        print(f"Successfully loaded and reshaped data.")
        print(f"Neighbor IDs shape: {neighbor_ids.shape}")
        print(f"Distances shape: {distances.shape}")
        
        # Print the first query's ground truth
        print("\nGround truth for Query 0:")
        for i in range(min(10, neighbors_per_query)): # Print top 10 or k
            print(f"  Rank {i}: ID={neighbor_ids[0, i]}, Dist={distances[0, i]:.4f}")

except FileNotFoundError:
    print("Error: sift10k_groundtruth.bin not found.")
except ValueError as e:
    print(f"Error reshaping ground truth data!")
    print(f"Original error: {e}")
    if 'data' in locals():
         print(f"Total records read: {len(data)}")
         print(f"Expected records (N*k): {expected_records}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Error reshaping ground truth data!
cannot reshape array of size 200000 into shape (10000,10)
